# 02 · Question 2 — Is the signal still valid?

Rolling, point-in-time monitoring in the vocabulary of bank model-risk reports
(`sv/validation/monitoring.py`). All metrics on date *t* use only the trailing 52 weeks ending the week before *t*.

| metric | what it watches | rule of thumb |
|---|---|---|
| `psi_score` | score distribution vs development sample (first 104 weeks) | < 0.10 stable · 0.10–0.25 watch · > 0.25 shifted |
| `psi_input` | realised-vol regime vs development sample | same |
| `auc_1w`, `ks_1w` | does a higher score still mean a higher next-week return? | AUC 0.5 = no skill |
| `auc_13w`, `calib_slope` | same at the models' stated 65-day horizon (lagged 13 weeks) | slope 1 = calibrated |
| `rolling_sharpe`, `active_drawdown` | portfolio-level health | — |

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from sv import db

con = db.connect(read_only=True)
q   = lambda name, **p: db.run_sql_file(con, name, p or None)   # run sql/queries/<name>.sql
sql = lambda text, **p: db.read(con, text, p or None)                # run an inline query
pd.set_option("display.width", 140); plt.rcParams["figure.figsize"] = (10, 4)

In [ ]:
mon = sql("SELECT date, model, metric, value FROM monitoring")
wide = {m: g.pivot(index="date", columns="metric", values="value") for m, g in mon.groupby("model")}
list(wide)

## Score stability (PSI) and discriminatory power (AUC) over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for m in ["momentum", "gbm_expected", "gbm"]:
    wide[m]["psi_score"].plot(ax=axes[0], label=m); wide[m]["auc_1w"].plot(ax=axes[1], label=m)
axes[0].axhline(0.25, color="r", ls="--"); axes[0].axhline(0.10, color="orange", ls=":"); axes[0].set_title("PSI of score vs development sample"); axes[0].legend()
axes[1].axhline(0.5, color="r", ls="--"); axes[1].set_title("Rolling 52w AUC for next-week direction"); axes[1].legend()
plt.tight_layout(); plt.show()

## Input regime: realised-volatility PSI (shared by all models)

In [ ]:
ax = wide["momentum"]["psi_input"].plot(title="PSI of trailing-252d realised vol vs development sample")
ax.axhline(0.25, color="r", ls="--"); plt.show()

## Calibration at the 65-day horizon

In [ ]:
fig, ax = plt.subplots()
for m in ["gbm_expected", "gbm"]:
    wide[m]["calib_slope"].plot(ax=ax, label=m)
ax.axhline(1, color="k", ls="--"); ax.axhline(0, color="r", ls=":"); ax.set_title("OLS slope of realised 13w return on predicted score (1 = calibrated)"); ax.legend(); plt.show()

## The same metrics, computed in SQL

Cross-check of the Python rolling metrics with set-based SQL (`sql/queries/auc_by_year.sql`,
`sql/queries/psi_score.sql`). AUC uses the Mann-Whitney rank identity, PSI uses decile buckets from the
development sample — no Python loops.

In [ ]:
pd.concat({m: q("auc_by_year", model=m).set_index("year")["auc"] for m in ["momentum", "gbm_expected", "gbm"]}, axis=1).round(3)

In [ ]:
pd.concat({m: q("psi_score", model=m, dev_weeks=104, buckets=config.PSI_BUCKETS).set_index("year")["psi"] for m in ["momentum", "gbm_expected", "gbm"]}, axis=1).round(3)

## Latest reading per model

In [ ]:
sql("""SELECT model, metric, ROUND(value, 3) AS value FROM monitoring
       WHERE date = (SELECT MAX(date) FROM monitoring) ORDER BY model, metric""").pivot(index="metric", columns="model", values="value")